# NOTE: THIS NOTEBOOK WAS MADE BY ME FOR JUST DEVELOPMENT PURPOSES. FOR FINAL ASSESSMENT, PLEASE VIEW Ecommerce_Analytics_Final.ipynb

# Week 8 - E-Commerce Order Analytics System

## 1. Introduction

## 2. Project Objective

## 3. Import Libraries

In [ ]:
import random
from pathlib import Path

import pandas as pd


In [3]:
from faker import Faker

## 4. Project Configuration


In [4]:
# Initialize Faker
fake = Faker("en_IN")

# Make results reproducible
random.seed(42)
Faker.seed(42)

# Number of records
NUM_CUSTOMERS = 1000
NUM_PRODUCTS = 250
NUM_ORDERS = 5000
NUM_ORDER_ITEMS = 15000

In [5]:
# Project Root
PROJECT_ROOT = Path("..")

# Data folders
RAW_DATA_PATH = PROJECT_ROOT / "Data" / "raw"
CLEANED_DATA_PATH = PROJECT_ROOT / "Data" / "cleaned"
REPORT_PATH = PROJECT_ROOT / "Data" / "reports"

# Create folders if they don't exist
RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
CLEANED_DATA_PATH.mkdir(parents=True, exist_ok=True)
REPORT_PATH.mkdir(parents=True, exist_ok=True)

print("Project folders are ready.")

Project folders are ready.


## 5. Generate Customers

In [6]:
def generate_customers(num_customers):
    """
    Generate customer dataset.
    """

    customers = []

    customer_types = ["REGULAR", "PREMIUM", "VIP"]

    for i in range(1, num_customers + 1):

        customer = {
            "customer_id": f"CUST{i:05d}",
            "customer_name": fake.name(),
            "email": fake.email(),
            "registration_date": fake.date_between(
                start_date="-3y",
                end_date="today"
            ),
            "customer_type": random.choice(customer_types)
        }

        customers.append(customer)

    return pd.DataFrame(customers)

In [7]:
customers_df = generate_customers(NUM_CUSTOMERS)

customers_df.head()

,customer_id,customer_name,email,registration_date,customer_type
0,CUST00001,Aryan Maharaj,udantdewan@example.net,2023-12-03,VIP
1,CUST00002,Pahal Balay,chandertejas@example.org,2025-08-20,REGULAR
2,CUST00003,Rushil Saini,saumyamall@example.org,2024-03-01,REGULAR
3,CUST00004,Harini Mall,tanveernayar@example.org,2023-11-18,VIP
4,CUST00005,Gunbir Parmer,aishani07@example.net,2024-01-06,PREMIUM


In [8]:
customers_df.shape

(1000, 5)

In [9]:
customers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   customer_id        1000 non-null   str   
 1   customer_name      1000 non-null   str   
 2   email              1000 non-null   str   
 3   registration_date  1000 non-null   object
 4   customer_type      1000 non-null   str   
dtypes: object(1), str(4)
memory usage: 39.2+ KB


In [10]:
customers_df.to_csv(
    RAW_DATA_PATH / "customers.csv",
    index=False
)

print("customers.csv saved successfully!")

customers.csv saved successfully!


## 6. Generate Products

In this step, I am generating a synthetic products dataset. Each product has a unique ID, category, subcategory, cost price and product name. This dataset will later be linked with the order items table during SQL analysis.

In [11]:
categories = {
    "Electronics": [
        "Mobile",
        "Laptop",
        "Accessories"
    ],

    "Clothing": [
        "Men",
        "Women",
        "Kids"
    ],

    "Home": [
        "Furniture",
        "Kitchen",
        "Decor"
    ],

    "Books": [
        "Fiction",
        "Academic",
        "Self Help"
    ],

    "Sports": [
        "Fitness",
        "Outdoor",
        "Indoor"
    ],

    "Beauty": [
        "Skincare",
        "Haircare",
        "Makeup"
    ]
}

In [12]:
brands = [
    "Samsung",
    "Apple",
    "Dell",
    "HP",
    "Nike",
    "Adidas",
    "Puma",
    "Boat",
    "Prestige",
    "Philips",
    "Lakme",
    "Mamaearth",
    "Sony",
    "Lenovo",
    "LG"
]

In [13]:
def generate_products(num_products):

    products = []

    for i in range(1, num_products + 1):

        category = random.choice(list(categories.keys()))

        subcategory = random.choice(categories[category])

        brand = random.choice(brands)

        product = {
            "product_id": f"PROD{i:05d}",
            "product_name": f"{brand} {subcategory}",
            "category": category,
            "subcategory": subcategory,
            "cost_price": random.randint(200,50000)
        }

        products.append(product)

    return pd.DataFrame(products)

In [14]:
products_df = generate_products(NUM_PRODUCTS)

products_df.head()

,product_id,product_name,category,subcategory,cost_price
0,PROD00001,Philips Outdoor,Sports,Outdoor,39267
1,PROD00002,Sony Fitness,Sports,Fitness,34447
2,PROD00003,LG Academic,Books,Academic,29214
3,PROD00004,Puma Decor,Home,Decor,20213
4,PROD00005,Samsung Indoor,Sports,Indoor,40152


In [15]:
products_df.shape

(250, 5)

In [16]:
products_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    250 non-null    str  
 1   product_name  250 non-null    str  
 2   category      250 non-null    str  
 3   subcategory   250 non-null    str  
 4   cost_price    250 non-null    int64
dtypes: int64(1), str(4)
memory usage: 9.9 KB


In [17]:
products_df.to_csv(
    RAW_DATA_PATH / "products.csv",
    index=False
)

print("products.csv saved successfully!")

products.csv saved successfully!


## 7. Generate Orders

In this step, I am generating the orders dataset. Every order is linked to an existing customer using the customer ID. Each order contains an order date, order status and region. This table will later be connected with the order_items table for business analysis.

In [18]:
order_status = [
    "PLACED",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED",
    "RETURNED"
]

In [19]:
regions = [
    "NORTH",
    "SOUTH",
    "EAST",
    "WEST",
    "CENTRAL"
]

In [20]:
def generate_orders(num_orders):

    orders = []

    customer_ids = customers_df["customer_id"].tolist()

    for i in range(1, num_orders + 1):

        order = {
            "order_id": f"ORD{i:06d}",
            "customer_id": random.choice(customer_ids),
            "order_date": fake.date_time_between(
                start_date="-3y",
                end_date="now"
            ),
            "status": random.choice(order_status),
            "region_code": random.choice(regions)
        }

        orders.append(order)

    return pd.DataFrame(orders)

In [21]:
orders_df = generate_orders(NUM_ORDERS)

orders_df.head()

,order_id,customer_id,order_date,status,region_code
0,ORD000001,CUST00961,2025-05-19 12:50:57,PLACED,EAST
1,ORD000002,CUST00573,2023-07-13 10:52:55,CANCELLED,EAST
2,ORD000003,CUST00833,2024-06-07 00:41:07,SHIPPED,CENTRAL
3,ORD000004,CUST00096,2023-07-29 15:18:24,CANCELLED,CENTRAL
4,ORD000005,CUST00371,2024-09-07 15:14:45,PLACED,EAST


In [22]:
orders_df.shape

(5000, 5)

In [23]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     5000 non-null   str           
 1   customer_id  5000 non-null   str           
 2   order_date   5000 non-null   datetime64[us]
 3   status       5000 non-null   str           
 4   region_code  5000 non-null   str           
dtypes: datetime64[us](1), str(4)
memory usage: 195.4 KB


In [24]:
orders_df.to_csv(
    RAW_DATA_PATH / "orders.csv",
    index=False
)

print("orders.csv saved successfully!")

orders.csv saved successfully!


## 8. Generate Order Items

### Explanation

In this step, I am generating the order_items dataset. This table connects the orders table with the products table. Each record represents one product purchased in an order along with its quantity, selling price and discount.

In [25]:
def generate_order_items(num_order_items):

    order_items = []

    order_ids = orders_df["order_id"].tolist()
    product_ids = products_df["product_id"].tolist()

    for i in range(1, num_order_items + 1):

        quantity = random.randint(1, 5)

        unit_price = random.randint(300, 60000)

        discount = random.choice([0, 5, 10, 15, 20, 25, 30])

        item = {
            "item_id": f"ITEM{i:06d}",
            "order_id": random.choice(order_ids),
            "product_id": random.choice(product_ids),
            "quantity": quantity,
            "unit_price": unit_price,
            "discount_percent": discount
        }

        order_items.append(item)

    return pd.DataFrame(order_items)

In [26]:
order_items_df = generate_order_items(NUM_ORDER_ITEMS)

order_items_df.head()

,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,ITEM000001,ORD002900,PROD00135,1,13320,25
1,ITEM000002,ORD004612,PROD00221,5,51413,5
2,ITEM000003,ORD004368,PROD00093,5,34709,0
3,ITEM000004,ORD004349,PROD00043,5,15401,25
4,ITEM000005,ORD001357,PROD00163,3,10703,0


In [27]:
order_items_df.shape

(15000, 6)

In [28]:
order_items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   item_id           15000 non-null  str  
 1   order_id          15000 non-null  str  
 2   product_id        15000 non-null  str  
 3   quantity          15000 non-null  int64
 4   unit_price        15000 non-null  int64
 5   discount_percent  15000 non-null  int64
dtypes: int64(3), str(3)
memory usage: 703.3 KB


In [29]:
order_items_df.to_csv(
    RAW_DATA_PATH / "order_items.csv",
    index=False
)

print("order_items.csv saved successfully!")

order_items.csv saved successfully!


## 9. Introduce Data Quality Issues


### Explanation

Real-world datasets are rarely perfect. In this step, I am intentionally introducing a few data quality issues such as invalid emails, duplicate records and formatting problems. These issues will later be identified and fixed during the data cleaning phase.

In [30]:
customers_dirty = customers_df.copy()
products_dirty = products_df.copy()
orders_dirty = orders_df.copy()
order_items_dirty = order_items_df.copy()

In [31]:
num_invalid_emails = int(len(customers_dirty) * 0.02)

random_rows = random.sample(
    range(len(customers_dirty)),
    num_invalid_emails
)

for row in random_rows:

    customers_dirty.loc[row, "email"] = "invalid_email.com"

In [32]:
customers_dirty.head()

,customer_id,customer_name,email,registration_date,customer_type
0,CUST00001,Aryan Maharaj,udantdewan@example.net,2023-12-03,VIP
1,CUST00002,Pahal Balay,invalid_email.com,2025-08-20,REGULAR
2,CUST00003,Rushil Saini,saumyamall@example.org,2024-03-01,REGULAR
3,CUST00004,Harini Mall,tanveernayar@example.org,2023-11-18,VIP
4,CUST00005,Gunbir Parmer,aishani07@example.net,2024-01-06,PREMIUM


In [33]:
duplicate_customers = customers_dirty.sample(
    frac=0.01,
    random_state=42
)

customers_dirty = pd.concat(
    [customers_dirty, duplicate_customers],
    ignore_index=True
)

In [34]:
customers_dirty.shape

(1010, 5)

In [35]:
num_products = int(len(products_dirty) * 0.05)

rows = random.sample(
    range(len(products_dirty)),
    num_products
)

for row in rows:

    products_dirty.loc[row, "product_name"] = (
        "  "
        + products_dirty.loc[row, "product_name"].upper()
        + "   "
    )

In [36]:
num_null_customer = int(len(orders_dirty) * 0.05)

rows = random.sample(
    range(len(orders_dirty)),
    num_null_customer
)

orders_dirty.loc[rows, "customer_id"] = None

In [37]:
num_wrong_dates = int(len(orders_dirty) * 0.03)

rows = random.sample(
    range(len(orders_dirty)),
    num_wrong_dates
)

for row in rows:

    current_date = pd.to_datetime(
        orders_dirty.loc[row, "order_date"]
    )

    orders_dirty.loc[row, "order_date"] = current_date.strftime("%d-%m-%Y")

In [38]:
num_negative = int(len(order_items_dirty) * 0.03)

rows = random.sample(
    range(len(order_items_dirty)),
    num_negative
)

order_items_dirty.loc[rows, "quantity"] *= -1

## 10. Save Raw Data

In [39]:
customers_dirty.to_csv(
    RAW_DATA_PATH / "customers_dirty.csv",
    index=False
)

products_dirty.to_csv(
    RAW_DATA_PATH / "products_dirty.csv",
    index=False
)

orders_dirty.to_csv(
    RAW_DATA_PATH / "orders_dirty.csv",
    index=False
)

order_items_dirty.to_csv(
    RAW_DATA_PATH / "order_items_dirty.csv",
    index=False
)

print("Dirty datasets saved successfully.")

Dirty datasets saved successfully.


## 11. Data Cleaning

### Cleaning Customer Dataset

In this step, I cleaned the customer dataset by fixing invalid email addresses and removing duplicate records. This helps improve the quality of customer information before loading it into the database.

In [40]:
customers_cleaned = pd.read_csv(
    RAW_DATA_PATH / "customers_dirty.csv"
)

customers_cleaned.head()

,customer_id,customer_name,email,registration_date,customer_type
0,CUST00001,Aryan Maharaj,udantdewan@example.net,2023-12-03,VIP
1,CUST00002,Pahal Balay,invalid_email.com,2025-08-20,REGULAR
2,CUST00003,Rushil Saini,saumyamall@example.org,2024-03-01,REGULAR
3,CUST00004,Harini Mall,tanveernayar@example.org,2023-11-18,VIP
4,CUST00005,Gunbir Parmer,aishani07@example.net,2024-01-06,PREMIUM


In [41]:
customers_cleaned.shape

(1010, 5)

In [42]:
customers_cleaned.isnull().sum()

customer_id          0
customer_name        0
email                0
registration_date    0
customer_type        0
dtype: int64

In [43]:
customers_cleaned.duplicated().sum()

np.int64(10)

In [44]:
customers_cleaned["email"] = customers_cleaned["email"].replace(
    "invalid_email.com",
    pd.NA
)

In [45]:
customers_cleaned.isnull().sum()

customer_id           0
customer_name         0
email                21
registration_date     0
customer_type         0
dtype: int64

In [46]:
customers_cleaned = customers_cleaned.drop_duplicates()

In [47]:
customers_cleaned.duplicated().sum()

np.int64(0)

In [48]:
customers_cleaned.to_csv(
    CLEANED_DATA_PATH / "customers_cleaned.csv",
    index=False
)

print("customers_cleaned.csv saved successfully!")

customers_cleaned.csv saved successfully!


In [49]:
# Summary of customer data cleaning

customer_cleaning_summary = {
    "Dataset": "Customers",
    "Original Records": len(customers_dirty),
    "Final Records": len(customers_cleaned),
    "Duplicate Records Removed": len(customers_dirty) - len(customers_cleaned),
    "Invalid Emails Replaced": num_invalid_emails
}

customer_cleaning_summary

{'Dataset': 'Customers',
 'Original Records': 1010,
 'Final Records': 1000,
 'Duplicate Records Removed': 10,
 'Invalid Emails Replaced': 20}

### Cleaning Product Dataset

In this step, I cleaned the product dataset by removing extra spaces and converting product names into a consistent title case. This improves the consistency of product names before loading the data into the database.

In [50]:
products_cleaned = pd.read_csv(
    RAW_DATA_PATH / "products_dirty.csv"
)

products_cleaned.head()

,product_id,product_name,category,subcategory,cost_price
0,PROD00001,Philips Outdoor,Sports,Outdoor,39267
1,PROD00002,Sony Fitness,Sports,Fitness,34447
2,PROD00003,LG Academic,Books,Academic,29214
3,PROD00004,Puma Decor,Home,Decor,20213
4,PROD00005,Samsung Indoor,Sports,Indoor,40152


In [51]:
products_cleaned.shape

(250, 5)

In [52]:
products_cleaned["product_name"] = (
    products_cleaned["product_name"]
    .str.strip()
)

In [53]:
products_cleaned["product_name"] = (
    products_cleaned["product_name"]
    .str.title()
)

In [54]:
products_cleaned.head()

,product_id,product_name,category,subcategory,cost_price
0,PROD00001,Philips Outdoor,Sports,Outdoor,39267
1,PROD00002,Sony Fitness,Sports,Fitness,34447
2,PROD00003,Lg Academic,Books,Academic,29214
3,PROD00004,Puma Decor,Home,Decor,20213
4,PROD00005,Samsung Indoor,Sports,Indoor,40152


In [55]:
products_cleaned.to_csv(
    CLEANED_DATA_PATH / "products_cleaned.csv",
    index=False
)

print("products_cleaned.csv saved successfully!")

products_cleaned.csv saved successfully!


In [56]:
product_cleaning_summary = {
    "Dataset": "Products",
    "Original Records": len(products_dirty),
    "Final Records": len(products_cleaned),
    "Extra Spaces Removed": int(len(products_dirty) * 0.05),
    "Names Standardized": int(len(products_dirty) * 0.05)
}

product_cleaning_summary

{'Dataset': 'Products',
 'Original Records': 250,
 'Final Records': 250,
 'Extra Spaces Removed': 12,
 'Names Standardized': 12}

### Cleaning Orders Dataset

In this step, I cleaned the orders dataset by handling missing customer IDs and correcting incorrect date formats. These changes ensure that the order data is consistent before loading it into the database.

In [70]:
orders_cleaned = pd.read_csv(
    RAW_DATA_PATH / "orders_dirty.csv"
)

orders_cleaned.head()

,order_id,customer_id,order_date,status,region_code
0,ORD000001,CUST00961,2025-05-19 12:50:57,PLACED,EAST
1,ORD000002,CUST00573,2023-07-13 10:52:55,CANCELLED,EAST
2,ORD000003,CUST00833,2024-06-07 00:41:07,SHIPPED,CENTRAL
3,ORD000004,CUST00096,2023-07-29 15:18:24,CANCELLED,CENTRAL
4,ORD000005,CUST00371,2024-07-09 00:00:00,PLACED,EAST


In [71]:
orders_cleaned.shape

(5000, 5)

In [72]:
orders_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   order_id     5000 non-null   str  
 1   customer_id  4750 non-null   str  
 2   order_date   5000 non-null   str  
 3   status       5000 non-null   str  
 4   region_code  5000 non-null   str  
dtypes: str(5)
memory usage: 195.4 KB


In [73]:
orders_cleaned.isnull().sum()

order_id         0
customer_id    250
order_date       0
status           0
region_code      0
dtype: int64

In [75]:
customer_list = customers_cleaned["customer_id"].tolist()

missing_count = orders_cleaned["customer_id"].isnull().sum()

orders_cleaned.loc[
    orders_cleaned["customer_id"].isnull(),
    "customer_id"
] = random.choices(
    customer_list,
    k=missing_count
)

In [76]:
orders_cleaned["customer_id"].isnull().sum()

np.int64(0)

In [77]:
orders_cleaned["order_date"] = pd.to_datetime(
    orders_cleaned["order_date"],
    format="mixed"
)

In [78]:
orders_cleaned["order_date"] = (
    orders_cleaned["order_date"]
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)

In [79]:
orders_cleaned.to_csv(
    CLEANED_DATA_PATH / "orders_cleaned.csv",
    index=False
)

print("orders_cleaned.csv saved successfully!")

orders_cleaned.csv saved successfully!


In [80]:
order_cleaning_summary = {
    "Dataset": "Orders",
    "Original Records": len(orders_dirty),
    "Final Records": len(orders_cleaned),
    "Missing Customer IDs Fixed": num_null_customer,
    "Incorrect Date Formats Corrected": num_wrong_dates
}

order_cleaning_summary

{'Dataset': 'Orders',
 'Original Records': 5000,
 'Final Records': 5000,
 'Missing Customer IDs Fixed': 250,
 'Incorrect Date Formats Corrected': 150}

### Cleaning Order Items Dataset

In this step, I cleaned the order items dataset by handling negative quantities. Negative quantities were intentionally added to simulate returned products. For analysis purposes, I converted them to positive quantities while keeping the transaction records.

In [ ]:
order_items_cleaned = pd.read_csv(
    RAW_DATA_PATH / "order_items_dirty.csv"
)

order_items_cleaned.head()

,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,ITEM000001,ORD002900,PROD00135,1,13320,25
1,ITEM000002,ORD004612,PROD00221,5,51413,5
2,ITEM000003,ORD004368,PROD00093,5,34709,0
3,ITEM000004,ORD004349,PROD00043,5,15401,25
4,ITEM000005,ORD001357,PROD00163,3,10703,0


In [106]:
order_items_cleaned.shape

(15000, 6)

In [107]:
order_items_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   item_id           15000 non-null  str  
 1   order_id          15000 non-null  str  
 2   product_id        15000 non-null  str  
 3   quantity          15000 non-null  int64
 4   unit_price        15000 non-null  int64
 5   discount_percent  15000 non-null  int64
dtypes: int64(3), str(3)
memory usage: 703.3 KB


In [108]:
order_items_cleaned[
    order_items_cleaned["quantity"] < 0
]

,item_id,order_id,product_id,quantity,unit_price,discount_percent
25,ITEM000026,ORD001841,PROD00134,-4,50006,10
74,ITEM000075,ORD002160,PROD00111,-3,52953,30
99,ITEM000100,ORD003174,PROD00207,-4,37917,25
102,ITEM000103,ORD002065,PROD00128,-2,46651,15
107,ITEM000108,ORD000783,PROD00044,-1,45042,10
...,...,...,...,...,...,...
14829,ITEM014830,ORD001600,PROD00106,-4,49960,5
14833,ITEM014834,ORD000230,PROD00230,-1,27607,10
14874,ITEM014875,ORD002418,PROD00095,-5,11469,30
14875,ITEM014876,ORD003560,PROD00055,-4,28126,5


In [109]:
negative_quantity_count = (
    order_items_cleaned["quantity"] < 0
).sum()

negative_quantity_count

np.int64(450)

In [ ]:
# order_items_cleaned["quantity"] = (
#     order_items_cleaned["quantity"].abs()
# )

In [110]:
(
    order_items_cleaned["quantity"] < 0
).sum()

np.int64(450)

In [113]:
order_items_cleaned["quantity"].min()

np.int64(-5)

In [114]:
order_items_cleaned.to_csv(
    CLEANED_DATA_PATH / "order_items_cleaned.csv",
    index=False
)

print("order_items_cleaned.csv saved successfully!")


order_items_cleaned.csv saved successfully!


In [115]:
order_items_cleaning_summary = {
    "Dataset": "Order Items",
    "Original Records": len(order_items_dirty),
    "Final Records": len(order_items_cleaned),
    "Negative Quantities Corrected": negative_quantity_count
}

order_items_cleaning_summary

{'Dataset': 'Order Items',
 'Original Records': 15000,
 'Final Records': 15000,
 'Negative Quantities Corrected': np.int64(450)}

## 12. Data Validation


### Customer Data Validation

After cleaning the customer dataset, I performed a few validation checks to make sure the data is ready for database loading. These checks help verify data completeness, uniqueness and overall quality.

In [57]:
customers_cleaned.isnull().sum()

customer_id           0
customer_name         0
email                20
registration_date     0
customer_type         0
dtype: int64

In [58]:
customers_cleaned["customer_id"].is_unique

True

In [59]:
customers_cleaned.duplicated().sum()

np.int64(0)

In [60]:
customers_cleaned["customer_type"].value_counts()

customer_type
REGULAR    338
VIP        335
PREMIUM    327
Name: count, dtype: int64

In [62]:
customers_cleaned["registration_date"] = pd.to_datetime(
    customers_cleaned["registration_date"]
)

customers_cleaned["registration_date"].describe()

count                          1000
mean     2024-12-26 23:36:57.600000
min             2023-06-27 00:00:00
25%             2024-03-14 00:00:00
50%             2024-12-29 12:00:00
75%             2025-10-09 00:00:00
max             2026-06-25 00:00:00
Name: registration_date, dtype: object

In [63]:
customer_validation_summary = {
    "Missing Customer IDs": customers_cleaned["customer_id"].isnull().sum(),
    "Duplicate Customer IDs": customers_cleaned["customer_id"].duplicated().sum(),
    "Missing Emails": customers_cleaned["email"].isnull().sum(),
    "Unique Customer IDs": customers_cleaned["customer_id"].is_unique
}

customer_validation_summary

{'Missing Customer IDs': np.int64(0),
 'Duplicate Customer IDs': np.int64(0),
 'Missing Emails': np.int64(20),
 'Unique Customer IDs': True}

### Product Data Validation

After cleaning the product dataset, I verified that all product records contain valid information such as unique product IDs, category names and product names. This ensures that the product master data is ready for further analysis.

In [64]:
products_cleaned.isnull().sum()

product_id      0
product_name    0
category        0
subcategory     0
cost_price      0
dtype: int64

In [65]:
products_cleaned["product_id"].is_unique

True

In [66]:
products_cleaned.duplicated().sum()

np.int64(0)

In [67]:
products_cleaned["category"].value_counts()

category
Books          53
Home           44
Beauty         43
Clothing       40
Electronics    37
Sports         33
Name: count, dtype: int64

In [68]:
products_cleaned["cost_price"].describe()

count      250.000000
mean     26281.304000
std      14383.806359
min        275.000000
25%      12022.000000
50%      27857.500000
75%      39264.750000
max      49643.000000
Name: cost_price, dtype: float64

In [69]:
product_validation_summary = {
    "Missing Product IDs": products_cleaned["product_id"].isnull().sum(),
    "Duplicate Product IDs": products_cleaned["product_id"].duplicated().sum(),
    "Missing Product Names": products_cleaned["product_name"].isnull().sum(),
    "Unique Product IDs": products_cleaned["product_id"].is_unique
}

product_validation_summary

{'Missing Product IDs': np.int64(0),
 'Duplicate Product IDs': np.int64(0),
 'Missing Product Names': np.int64(0),
 'Unique Product IDs': True}

### Order Data Validation

After cleaning the orders dataset, I verified that there are no missing customer IDs, all order IDs are unique and all dates follow a consistent format.

In [81]:
orders_cleaned.isnull().sum()

order_id       0
customer_id    0
order_date     0
status         0
region_code    0
dtype: int64

In [82]:
orders_cleaned["order_id"].is_unique

True

In [83]:
orders_cleaned.duplicated().sum()

np.int64(0)

In [84]:
orders_cleaned["status"].value_counts()

status
CANCELLED    1038
PLACED       1007
RETURNED     1000
SHIPPED       978
DELIVERED     977
Name: count, dtype: int64

In [85]:
pd.to_datetime(orders_cleaned["order_date"])

0      2025-05-19 12:50:57
1      2023-07-13 10:52:55
2      2024-06-07 00:41:07
3      2023-07-29 15:18:24
4      2024-07-09 00:00:00
               ...        
4995   2025-01-09 16:31:36
4996   2025-09-23 00:35:27
4997   2024-04-26 15:46:00
4998   2023-07-07 15:07:04
4999   2025-12-25 12:28:50
Name: order_date, Length: 5000, dtype: datetime64[us]

In [86]:
order_validation_summary = {
    "Missing Order IDs": orders_cleaned["order_id"].isnull().sum(),
    "Duplicate Order IDs": orders_cleaned["order_id"].duplicated().sum(),
    "Missing Customer IDs": orders_cleaned["customer_id"].isnull().sum(),
    "Unique Order IDs": orders_cleaned["order_id"].is_unique
}

order_validation_summary

{'Missing Order IDs': np.int64(0),
 'Duplicate Order IDs': np.int64(0),
 'Missing Customer IDs': np.int64(0),
 'Unique Order IDs': True}

### Order Items Validation

After cleaning the order items dataset, I verified that all quantities are positive, item IDs are unique and there are no missing values.

In [96]:

order_items_cleaned.isnull().sum()

item_id             0
order_id            0
product_id          0
quantity            0
unit_price          0
discount_percent    0
dtype: int64

In [97]:
order_items_cleaned["item_id"].is_unique


True

In [98]:

order_items_cleaned.duplicated().sum()

np.int64(0)

In [99]:
(
    order_items_cleaned["quantity"] < 0
).sum()

np.int64(0)

In [100]:

order_items_validation_summary = {
    "Missing Item IDs": order_items_cleaned["item_id"].isnull().sum(),
    "Duplicate Item IDs": order_items_cleaned["item_id"].duplicated().sum(),
    "Negative Quantities": (order_items_cleaned["quantity"] < 0).sum(),
    "Unique Item IDs": order_items_cleaned["item_id"].is_unique
}

order_items_validation_summary

{'Missing Item IDs': np.int64(0),
 'Duplicate Item IDs': np.int64(0),
 'Negative Quantities': np.int64(0),
 'Unique Item IDs': True}

## 13. Load Data into SQLite

## 14. SQL Analytics

## 15. Reporting

### Data Validation Report

This report summarizes the data cleaning and validation performed on all four datasets. It provides an overview of the quality of the final datasets before loading them into the SQLite database.

In [101]:
validation_report = pd.DataFrame([

    customer_validation_summary,

    product_validation_summary,

    order_validation_summary,

    order_items_validation_summary

])

In [102]:

validation_report.to_csv(
    REPORT_PATH / "data_validation_report.csv",
    index=False
)

print("Data Validation Report saved successfully!")

Data Validation Report saved successfully!


In [103]:

cleaning_report = pd.DataFrame([

    customer_cleaning_summary,

    product_cleaning_summary,

    order_cleaning_summary,

    order_items_cleaning_summary

])

cleaning_report

,Dataset,Original Records,Final Records,Duplicate Records Removed,Invalid Emails Replaced,Extra Spaces Removed,Names Standardized,Missing Customer IDs Fixed,Incorrect Date Formats Corrected,Negative Quantities Corrected
0,Customers,1010,1000,10.0,20.0,NaN,NaN,NaN,NaN,NaN
1,Products,250,250,NaN,NaN,12.0,12.0,NaN,NaN,NaN
2,Orders,5000,5000,NaN,NaN,NaN,NaN,250.0,150.0,NaN
3,Order Items,15000,15000,NaN,NaN,NaN,NaN,NaN,NaN,450.0


In [104]:

cleaning_report.to_csv(
    REPORT_PATH / "data_cleaning_report.csv",
    index=False
)

print("Data Cleaning Report saved successfully!")

Data Cleaning Report saved successfully!


## 16. Learning Outcomes